# Stages 0–4 check — download → frames → gaze projection → DINOv2 → visual verification

Runs the pipeline on **one** AEA sequence, with `WANT_VRS = True` so the accurate
calibration projection is exercised, then **verifies that the gaze (x, y) coordinates are
actually correct** and that the two similarity signals behave sensibly.

All pipeline code is copied verbatim from the repo, so no clone is needed.

| Step | What it does |
|---|---|
| 1–6 | **Stage 0** — download the mp4 + gaze zip + VRS, verify what landed |
| 7–8 | gaze CSV sanity, video sanity, clock agreement |
| 9 | **Stage 1** — subsample to 1 FPS |
| 10 | **Stage 2a** — build both projectors: OLD (buggy) and NEW (fixed) |
| T1 | **oracle diff** vs Meta's `get_gaze_vector_reprojection`, in native coords |
| T2 | how large was the transform bug, in pixels |
| T3 | **axis sweep** — catches swapped/rotated axes with no ground truth |
| T4 | distribution sanity — clipping, out-of-FOV rate |
| T5 | **visual overlay** — does the dot land on what the wearer is looking at |
| T5b | **four candidate rotations side by side** — confirms the rotation direction |
| T6 | scanpath over a single frame |
| 11 | **Stage 2b/3** — DINOv2 encode, gaze patch cell, pairwise similarities |
| S4a | **Stage 4** — scatter of all pairs split into the four gate quadrants |
| S4b | the four **extremes**: most / least similar frame, most / least similar gaze token |
| S4c | one example pair per **gate quadrant** |
| 12 | verdict summary |

## The two projection defects under test

**(a) Missing CPF → camera transform.** The original code fed a point in **CPF** (Central
Pupil Frame, between the eyes) straight into `cam_calib.project()`, which expects a point
already in the **camera** frame. Measured on a real sequence this was worth **~277 px on a
1408 px image — 1.4 DINOv2 cells**, i.e. large enough to select the wrong gaze token
outright. The fix uses the transform chain from Meta's own AEA quickstart tutorial.

**(b) Native vs upright orientation.** Aria's RGB sensor stores frames rotated. Meta's
tutorial projects onto the raw VRS image and so lands in that native frame; this pipeline
uses the **upright** preview MP4, so the result must be rotated by the coordinate
equivalent of `np.rot90(img, k=3)`. The frames are **square**, so no size or aspect-ratio
check can see this — an axis sweep (T3) or a visual overlay (T5/T5b) is required.

A first run produced `T1 = 31 px residual` and `T3 = FAIL (axes swapped)`, which motivated
both fixes. The second run gave `T1 = 10 px` (0.05 of a cell — immaterial) and `T3 = PASS`,
with T5b confirming `cw90` visually.

## 1 — Setup

In [ ]:
!pip -q install opencv-python-headless pandas numpy projectaria-tools

import os, glob, json, zipfile, urllib.request, time
import numpy as np, pandas as pd, cv2
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

import projectaria_tools
print("opencv             :", cv2.__version__)
print("projectaria_tools  :", getattr(projectaria_tools, "__version__", "(no __version__ attr)"))

## 2 — Load the download-links JSON

Links expire after ~14 days — re-download from
[projectaria.com/datasets/aea](https://www.projectaria.com/datasets/aea/) if downloads fail.

In [ ]:
# ---- Option A: upload from your laptop (default) ----
from google.colab import files
up = files.upload()                 # pick your aea_download_urls.json
URLS_JSON = list(up.keys())[0]

# ---- Option B: already in Google Drive ----
# from google.colab import drive; drive.mount('/content/drive')
# URLS_JSON = "/content/drive/MyDrive/aea/aea_download_urls.json"

meta = json.load(open(URLS_JSON))

print("top-level keys :", list(meta))
print("dataset        :", meta.get("sequence_config", {}).get("dataset_name"))
print("release        :", meta.get("sequence_config", {}).get("release"))
print("n sequences    :", len(meta["sequences"]))

## 3 — Pick ONE sequence

In [ ]:
seqs = list(meta["sequences"])
print("first 5 sequence names:")
for s in seqs[:5]:
    print("   ", s)

# >>> change this index to test a different sequence <<<
SEQ = seqs[0]

e = meta["sequences"][SEQ]
print(f"\nentries available for {SEQ}:")
for k in sorted(e):
    mb = e[k].get("file_size_bytes", 0) / 1e6
    print(f"   {k:24s} {mb:9.1f} MB   {e[k].get('filename','')}")

required = ["video_main_rgb", "mps_eye_gaze", "main_vrs"]
missing = [k for k in required if k not in e]
print("\nrequired keys:", "ALL PRESENT" if not missing else f"MISSING -> {missing}")

## 4 — Configure

`WANT_VRS = True` this time. The VRS is ~2 GB and is the slow part — expect several
minutes. It is required: the calibration projection and the whole point of this
notebook depend on it.

In [ ]:
RAW_DIR  = "/content/aea_check/raw"
WANT_VRS = True         # needed for calibration projection + IMU

need  = ["video_main_rgb", "mps_eye_gaze"] + (["main_vrs"] if WANT_VRS else [])
total = sum(e[k].get("file_size_bytes", 0) for k in need)

print(f"sequence : {SEQ}")
print(f"files    : {need}")
print(f"total    : {total/1e6:,.0f} MB   <- the VRS dominates this")

import shutil
free = shutil.disk_usage("/content").free / 1e9
print(f"free disk: {free:.1f} GB   {'OK' if free > total/1e9 + 2 else '!! LOW'}")

## 5 — Stage 0: acquisition

Verbatim from `src/common/aria_io.py:5-37`, progress hook added. Re-running is free:
the `os.path.exists` guards skip anything already downloaded.

In [ ]:
def _progress(name):
    t0 = time.time()
    def hook(blocks, bs, total):
        done = blocks * bs
        pct  = 100 * done / total if total > 0 else 0
        print(f"\r   {name:16s} {done/1e6:8.1f} MB  {min(pct,100):5.1f}%  ({time.time()-t0:4.0f}s)", end="")
    return hook


# ---- verbatim from src/common/aria_io.py (progress hook added) ----
def download_sequence(seq, meta, raw_dir, want_calibration=False):
    seq_dir = os.path.join(raw_dir, seq)
    os.makedirs(os.path.join(seq_dir, "eye_gaze"), exist_ok=True)
    e = meta["sequences"][seq]

    rgb = e["video_main_rgb"]
    mp4 = os.path.join(seq_dir, rgb["filename"])
    if not os.path.exists(mp4):
        urllib.request.urlretrieve(rgb["download_url"], mp4, _progress("video_main_rgb")); print()

    eg = e["mps_eye_gaze"]
    zp = os.path.join(seq_dir, eg["filename"])
    if not os.path.exists(zp):
        urllib.request.urlretrieve(eg["download_url"], zp, _progress("mps_eye_gaze")); print()
    with zipfile.ZipFile(zp) as z:
        z.extractall(os.path.join(seq_dir, "eye_gaze"))

    vrs_path = None
    if want_calibration:
        vrs = e.get("main_vrs")
        if vrs is None:
            raise RuntimeError(f"[{seq}] calibration/IMU requested but no 'main_vrs' entry in the URLs JSON.")
        vrs_path = os.path.join(seq_dir, vrs["filename"])
        if not os.path.exists(vrs_path):
            urllib.request.urlretrieve(vrs["download_url"], vrs_path, _progress("main_vrs")); print()
        if not os.path.exists(vrs_path):
            raise RuntimeError(f"[{seq}] VRS download failed; no file at {vrs_path}")

    return seq_dir, mp4, vrs_path


t0 = time.time()
seq_dir, mp4, vrs_path = download_sequence(SEQ, meta, RAW_DIR, want_calibration=WANT_VRS)
print(f"\ndone in {time.time()-t0:.0f}s\n")
print("seq_dir  :", seq_dir)
print("mp4      :", mp4)
print("vrs_path :", vrs_path)

## 6 — What landed on disk

In [ ]:
print(f"{SEQ}/")
rows = []
for root, dirs, fs in os.walk(seq_dir):
    for f in sorted(fs):
        p = os.path.join(root, f)
        rows.append((os.path.getsize(p)/1e6, os.path.relpath(p, seq_dir)))
for mb, rel in sorted(rows, key=lambda r: -r[0]):
    print(f"   {mb:9.2f} MB   {rel}")
print(f"\n   {sum(r[0] for r in rows):9.2f} MB   TOTAL")

hits = glob.glob(os.path.join(seq_dir, "eye_gaze", "**", "general_eye_gaze.csv"), recursive=True)
print("\ngeneral_eye_gaze.csv:", "FOUND" if hits else "!! MISSING -- unzip failed")
GAZE_CSV = hits[0] if hits else None

## 7 — Gaze CSV sanity

`load_gaze_raw` verbatim from `aria_io.py:40-46`.

In [ ]:
def load_gaze_raw(seq_dir):
    gcsv = glob.glob(os.path.join(seq_dir, "eye_gaze", "**", "general_eye_gaze.csv"),
                     recursive=True)[0]
    g = pd.read_csv(gcsv)
    ts = g["tracking_timestamp_us"].to_numpy(); ts = ts - ts[0]
    return ts, g["yaw_rads_cpf"].to_numpy(), g["pitch_rads_cpf"].to_numpy()


g_ts, g_yaw, g_pit = load_gaze_raw(seq_dir)
dts = np.diff(g_ts) / 1e6
dts = dts[dts > 0]
gaze_dur = g_ts[-1] / 1e6

print(f"samples      : {len(g_ts):,}")
print(f"duration     : {gaze_dur:.1f} s")
print(f"median dt    : {np.median(dts)*1000:.1f} ms   ->   ~{1/np.median(dts):.1f} Hz")
print(f"yaw  range   : [{g_yaw.min():+.3f}, {g_yaw.max():+.3f}] rad  ({np.rad2deg(g_yaw.min()):+.0f}..{np.rad2deg(g_yaw.max()):+.0f} deg)")
print(f"pitch range  : [{g_pit.min():+.3f}, {g_pit.max():+.3f}] rad  ({np.rad2deg(g_pit.min()):+.0f}..{np.rad2deg(g_pit.max()):+.0f} deg)")
print(f"NaNs         : yaw={np.isnan(g_yaw).sum()}  pitch={np.isnan(g_pit).sum()}")

fig, ax = plt.subplots(2, 1, figsize=(11, 4), sharex=True)
ax[0].plot(g_ts/1e6, g_yaw, lw=.6); ax[0].set_ylabel("yaw (rad)")
ax[1].plot(g_ts/1e6, g_pit, lw=.6); ax[1].set_ylabel("pitch (rad)"); ax[1].set_xlabel("t (s)")
fig.suptitle(f"raw gaze — {SEQ}"); plt.tight_layout(); plt.show()

## 8 — Video sanity + clock agreement

The pipeline synthesizes frame timestamps from the frame index and rebases gaze to its own
first sample, so it assumes both streams start together and run at the same rate.

In [ ]:
cap    = cv2.VideoCapture(mp4)
native = cap.get(cv2.CAP_PROP_FPS) or 20.0
n      = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
w_vid  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h_vid  = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
vid_dur = n / native

print(f"native fps   : {native:.2f}")
print(f"frame count  : {n:,}")
print(f"resolution   : {w_vid} x {h_vid}")
print(f"duration     : {vid_dur:.1f} s")

step = max(1, int(round(native / 1.0)))
kept = len(range(0, n, step))
print(f"\nat target_fps=1.0  ->  step={step}, ~{kept} frames kept, ~{kept-1} CSV rows")

print("\nCLOCK CHECK")
print(f"   video duration : {vid_dur:7.1f} s")
print(f"   gaze  duration : {gaze_dur:7.1f} s")
clock_gap = abs(vid_dur - gaze_dur)
print(f"   difference     : {clock_gap:7.1f} s   {'OK' if clock_gap < 1.0 else 'MISMATCH -- frame/gaze join is suspect'}")

ok, fr0 = cap.read()
cap.release()
plt.figure(figsize=(5,5))
plt.imshow(cv2.cvtColor(fr0, cv2.COLOR_BGR2RGB)); plt.axis("off")
plt.title(f"frame 0 — {SEQ}"); plt.show()

## 9 — Stage 1: subsample to 1 FPS

`subsample_frames` verbatim from `aria_io.py:106-121`. Capped to keep the run quick.

In [ ]:
MAX_SECONDS = 90        # None -> whole video
FRAMES_DIR  = "/content/aea_check/frames_1fps"

def subsample_frames(mp4, out_dir, target_fps=1.0, max_frames=None):
    os.makedirs(out_dir, exist_ok=True)
    cap = cv2.VideoCapture(mp4); native = cap.get(cv2.CAP_PROP_FPS) or 20.0
    step = max(1, int(round(native / target_fps)))
    paths, ts_us, kept, i = [], [], 0, 0
    while True:
        ok, fr = cap.read()
        if not ok or (max_frames is not None and i >= max_frames):
            break
        if i % step == 0:
            p = os.path.join(out_dir, f"frame_{kept:05d}.jpg")
            cv2.imwrite(p, fr)
            paths.append(p); ts_us.append((i / native) * 1e6); kept += 1
        i += 1
    cap.release()
    return paths, np.array(ts_us), native


mf = int(MAX_SECONDS * native) if MAX_SECONDS else None
paths, f_ts, native = subsample_frames(mp4, os.path.join(FRAMES_DIR, SEQ), 1.0, max_frames=mf)
print(f"kept {len(paths)} frames  ->  {len(paths)-1} consecutive pairs")
print("timestamps (s):", np.round(f_ts[:8]/1e6, 2))

## 10 — Stage 2a: build BOTH projectors

**Two independent defects are being corrected here.**

**(a) Missing CPF → camera transform.** `project_old` reproduces the original bug: a point
in CPF (Central Pupil Frame, between the eyes) fed straight into `cam_calib.project()`,
which expects a point already in the camera frame. `project_new` uses the exact chain from
Meta's AEA quickstart tutorial (`draw_eye_gaze`):

```python
gaze_cpf = get_eyegaze_point_at_depth(yaw, pitch, depth_m)
gaze_cam = T_device_camera.inverse() @ T_device_cpf @ gaze_cpf
px       = camera_calib.project(gaze_cam)
```

Note it uses `get_transform_device_cpf()` + `get_transform_device_camera()`, **not** the
`get_transform_cpf_sensor()` shortcut — the shortcut can return nominal CAD extrinsics
rather than the factory-calibrated ones.

**(b) Native vs upright orientation.** The tutorial projects onto the raw VRS image, which
is in the sensor's **native rotated** orientation. This pipeline decodes the **upright**
preview MP4, so the projected pixel must be rotated to match — the coordinate form of
`np.rot90(img, k=3)`. Because the frames are square, no size or aspect-ratio check can
detect this; T3 and T5b are what catch it.

In [ ]:
from projectaria_tools.core import data_provider
from projectaria_tools.core.mps import get_eyegaze_point_at_depth

STREAM_LABEL = "camera-rgb"
DEPTH_M      = 1.0
ROTATE_CW90  = True    # preview mp4 is upright; projected pixels are in the NATIVE frame

provider     = data_provider.create_vrs_data_provider(vrs_path)
device_calib = provider.get_device_calibration()
cam_calib    = device_calib.get_camera_calib(STREAM_LABEL)
W_cal, H_cal = cam_calib.get_image_size()

# transforms used by Meta's own AEA quickstart tutorial (`draw_eye_gaze`)
T_device_cpf = device_calib.get_transform_device_cpf()
T_device_cam = cam_calib.get_transform_device_camera()

print(f"calibration image size : {W_cal} x {H_cal}")
print(f"preview mp4 size       : {w_vid} x {h_vid}")
ar_cal, ar_vid = W_cal/H_cal, w_vid/h_vid
print(f"aspect ratios          : calib {ar_cal:.3f}  vs  video {ar_vid:.3f}   "
      f"{'OK' if abs(ar_cal-ar_vid) < 0.02 else '!! DIFFER -- possible rotation/crop'}")
print("   (square frames -> a 90 deg rotation is INVISIBLE to this check; see T3)")

try:
    M = np.array((T_device_cam.inverse() @ T_device_cpf).to_matrix())
    print(f"\nCPF->camera translation (m): {np.round(M[:3,3], 4)}")
    print(f"   |baseline| = {np.linalg.norm(M[:3,3])*100:.1f} cm  <- what the old code ignored")
except Exception as ex:
    print("\n(could not print transform matrix:", ex, ")")


def native_to_upright_cw90(x, y):
    """native sensor coords -> upright preview-mp4 coords (== np.rot90(img, k=3))"""
    return 1.0 - y, x


# ---------- OLD (original bug): CPF point fed straight to project(), no rotation ----------
def project_old(yaw, pitch, depth=DEPTH_M):
    pt_cpf = get_eyegaze_point_at_depth(yaw, pitch, depth)
    px = cam_calib.project(pt_cpf)
    if px is None:
        return 0.5, 0.5, True
    u, v = np.asarray(px, dtype=np.float64).flatten()[:2]
    return float(np.clip(u/W_cal, 0, 1)), float(np.clip(v/H_cal, 0, 1)), False


# ---------- NEW (fixed): official tutorial chain + upright rotation ----------
#   gaze_cam = T_device_camera.inverse() @ T_device_cpf @ gaze_cpf
def project_new(yaw, pitch, depth=DEPTH_M, rotate=None):
    rotate = ROTATE_CW90 if rotate is None else rotate
    gaze_cpf = get_eyegaze_point_at_depth(yaw, pitch, depth)
    gaze_cam = T_device_cam.inverse() @ T_device_cpf @ gaze_cpf
    px = cam_calib.project(gaze_cam)
    if px is None:
        return 0.5, 0.5, True
    u, v = np.asarray(px, dtype=np.float64).flatten()[:2]
    x, y = u / W_cal, v / H_cal                    # native sensor orientation
    if rotate:
        x, y = native_to_upright_cw90(x, y)
    return float(np.clip(x, 0, 1)), float(np.clip(y, 0, 1)), False


print("\nsmoke test at yaw=pitch=0 (straight ahead):")
print("   old            ->", np.round(project_old(0.0, 0.0)[:2], 4))
print("   new (native)   ->", np.round(project_new(0.0, 0.0, rotate=False)[:2], 4))
print("   new (upright)  ->", np.round(project_new(0.0, 0.0)[:2], 4))

## T1 — Oracle diff (the decisive test)

Compares both projectors against Meta's own `get_gaze_vector_reprojection`.
A correct implementation agrees to float noise.

**PASS** = `project_new` mean error ≈ 0.

In [ ]:
from projectaria_tools.core import mps
from projectaria_tools.core.mps.utils import get_gaze_vector_reprojection

eyegazes = mps.read_eyegaze(GAZE_CSV)
print(f"{len(eyegazes):,} EyeGaze records read by projectaria_tools")

N = min(3000, len(eyegazes))
ref, ref_oof = [], []
for g in eyegazes[:N]:
    px = get_gaze_vector_reprojection(g, STREAM_LABEL, device_calib, cam_calib, DEPTH_M)
    if px is None:
        ref.append((0.5, 0.5)); ref_oof.append(True)
    else:
        u, v = np.asarray(px, dtype=np.float64).flatten()[:2]
        ref.append((float(np.clip(u/W_cal,0,1)), float(np.clip(v/H_cal,0,1))))
        ref_oof.append(False)
ref     = np.array(ref)          # oracle is in NATIVE sensor coords
ref_oof = np.array(ref_oof)

# compare in native coords -- this isolates the transform chain from the rotation,
# which T3/T5 test separately
new = np.array([project_new(g.yaw, g.pitch, rotate=False)[:2] for g in eyegazes[:N]])
old = np.array([project_old(g.yaw, g.pitch)[:2] for g in eyegazes[:N]])

m = ~ref_oof
d_new = np.hypot(*(new - ref)[m].T)
d_old = np.hypot(*(old - ref)[m].T)

print(f"\ncompared on {m.sum():,} in-FOV samples (oracle out-of-FOV: {(~m).sum()})\n")
print(f"   NEW vs oracle : mean {d_new.mean():.8f}  max {d_new.max():.8f}   = {d_new.mean()*W_cal:8.3f} px")
print(f"   OLD vs oracle : mean {d_old.mean():.8f}  max {d_old.max():.8f}   = {d_old.mean()*W_cal:8.3f} px")

T1_PASS = d_new.mean() < 1e-5
print("\nT1:", "PASS -- transform chain matches Meta's helper exactly" if T1_PASS
      else "FAIL -- project_new still disagrees with the oracle; investigate")

# the rotation applied on top, for reference (T3/T5 verify it)
new_up = np.array([project_new(g.yaw, g.pitch)[:2] for g in eyegazes[:N]])
print(f"\nrotation ON adds a further {np.hypot(*(new_up - new)[m].T).mean()*W_cal:.1f} px of displacement")
print("   (expected: the 90 deg mapping is a large move, verified visually in T5)")

## T2 — How big was the bug?

Quantifies the old code's error in pixels, and shows whether it was a constant offset
(pure translation) or varied with gaze angle.

In [ ]:
off = (old - ref)[m]
print(f"OLD error, in pixels of a {W_cal}x{H_cal} image:")
print(f"   mean {d_old.mean()*W_cal:7.2f} px    median {np.median(d_old)*W_cal:7.2f} px    max {d_old.max()*W_cal:7.2f} px")
print(f"   mean signed dx {off[:,0].mean()*W_cal:+7.2f} px   (std {off[:,0].std()*W_cal:.2f})")
print(f"   mean signed dy {off[:,1].mean()*H_cal:+7.2f} px   (std {off[:,1].std()*H_cal:.2f})")
print(f"\n   as a fraction of image width: {d_old.mean()*100:.2f}%")
print(f"   as a fraction of a 7x7 DINOv2 cell (1 cell = {W_cal/7:.0f} px): {d_old.mean()*W_cal/(W_cal/7):.2f} cells")

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].hist(d_old*W_cal, bins=60, color="tab:red", alpha=.8)
ax[0].set_xlabel("OLD error (px)"); ax[0].set_ylabel("count"); ax[0].set_title("error magnitude")
ax[1].scatter(off[:,0]*W_cal, off[:,1]*H_cal, s=2, alpha=.3)
ax[1].axhline(0, lw=.5, c="k"); ax[1].axvline(0, lw=.5, c="k")
ax[1].set_xlabel("dx (px)"); ax[1].set_ylabel("dy (px)"); ax[1].set_title("error direction")
plt.tight_layout(); plt.show()

print("A tight cluster away from (0,0) = constant baseline offset.")
print("A wide spread = the error also depends on gaze angle (rotation component).")

## T3 — Axis sweep (catches rotated / swapped axes)

Needs no ground truth. Sweep yaw with pitch fixed at 0: **x must move, y must not**.
Then sweep pitch: **y must move, x must not**.

If the responses are swapped, the image is rotated 90° relative to the calibration —
the preview-MP4 orientation risk.

In [ ]:
ang = np.deg2rad(np.linspace(-25, 25, 51))

sweep_yaw   = np.array([project_new(a, 0.0)[:2] for a in ang])
sweep_pitch = np.array([project_new(0.0, a)[:2] for a in ang])

yx, yy = np.ptp(sweep_yaw[:,0]),   np.ptp(sweep_yaw[:,1])
px_, py = np.ptp(sweep_pitch[:,0]), np.ptp(sweep_pitch[:,1])

print(f"yaw   sweep (+-25 deg) -> x range {yx:.3f}   y range {yy:.3f}   (expect x >> y)")
print(f"pitch sweep (+-25 deg) -> x range {px_:.3f}   y range {py:.3f}   (expect y >> x)")

T3_PASS = (yx > 3*yy) and (py > 3*px_)
print("\nT3:", "PASS -- yaw drives x, pitch drives y" if T3_PASS else
      "FAIL -- axes look swapped or rotated; check preview-mp4 orientation vs calibration")

mono_x = np.all(np.diff(sweep_yaw[:,0]) > 0) or np.all(np.diff(sweep_yaw[:,0]) < 0)
mono_y = np.all(np.diff(sweep_pitch[:,1]) > 0) or np.all(np.diff(sweep_pitch[:,1]) < 0)
print(f"   monotonic x vs yaw   : {mono_x}")
print(f"   monotonic y vs pitch : {mono_y}   (both should be True)")
print(f"   x increases with yaw   : {sweep_yaw[-1,0]  > sweep_yaw[0,0]}")
print(f"   y increases with pitch : {sweep_pitch[-1,1] > sweep_pitch[0,1]}  "
      f"(False expected: image rows grow downward, pitch grows up)")

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(np.rad2deg(ang), sweep_yaw[:,0], label="x"); ax[0].plot(np.rad2deg(ang), sweep_yaw[:,1], label="y")
ax[0].set_title("yaw sweep (pitch=0)"); ax[0].set_xlabel("yaw (deg)"); ax[0].legend(); ax[0].set_ylim(0,1)
ax[1].plot(np.rad2deg(ang), sweep_pitch[:,0], label="x"); ax[1].plot(np.rad2deg(ang), sweep_pitch[:,1], label="y")
ax[1].set_title("pitch sweep (yaw=0)"); ax[1].set_xlabel("pitch (deg)"); ax[1].legend(); ax[1].set_ylim(0,1)
plt.tight_layout(); plt.show()

## T4 — Distribution sanity over the whole sequence

Catches silent failure modes: everything pinned at the 0.5 fallback, or `clip`
saturating at the borders.

In [ ]:
good = np.isfinite(g_yaw) & np.isfinite(g_pit)
out  = np.array([project_new(y, p) for y, p in zip(g_yaw[good], g_pit[good])], dtype=object)
xy   = np.array([[float(r[0]), float(r[1])] for r in out])
oof  = np.array([bool(r[2]) for r in out])

print(f"samples projected : {len(xy):,}")
print(f"out-of-FOV rate   : {oof.mean()*100:5.2f} %   {'OK' if oof.mean() < 0.15 else '!! HIGH'}")

iv = xy[~oof]
print(f"\nin-FOV x: mean {iv[:,0].mean():.3f}  std {iv[:,0].std():.3f}  min {iv[:,0].min():.3f}  max {iv[:,0].max():.3f}")
print(f"in-FOV y: mean {iv[:,1].mean():.3f}  std {iv[:,1].std():.3f}  min {iv[:,1].min():.3f}  max {iv[:,1].max():.3f}")

clipped = ((iv <= 0.0) | (iv >= 1.0)).any(axis=1).mean()
print(f"\nfraction hitting the clip border : {clipped*100:5.2f} %   {'OK' if clipped < 0.05 else '!! HIGH -- projection diverging'}")
print(f"fraction exactly at (0.5, 0.5)   : {np.isclose(xy, 0.5).all(axis=1).mean()*100:5.2f} %   (should ~= out-of-FOV rate)")

T4_PASS = (oof.mean() < 0.15) and (clipped < 0.05) and (iv[:,0].std() > 0.02) and (iv[:,1].std() > 0.02)
print("\nT4:", "PASS" if T4_PASS else "FAIL / SUSPICIOUS -- see flags above")

fig, ax = plt.subplots(1, 2, figsize=(11, 4.2))
ax[0].hist2d(iv[:,0], iv[:,1], bins=60, range=[[0,1],[0,1]], cmap="viridis")
ax[0].invert_yaxis(); ax[0].set_title("gaze density in image coords"); ax[0].set_xlabel("x"); ax[0].set_ylabel("y")
ax[1].plot(iv[:600,0], iv[:600,1], lw=.5, alpha=.8)
ax[1].set_xlim(0,1); ax[1].set_ylim(1,0); ax[1].set_title("first 600 samples (scanpath)")
plt.tight_layout(); plt.show()

## T5 — Visual overlay (the human check)

The numeric tests prove the code matches Meta's helper. This proves the result actually
lands on what the wearer was looking at — the one thing no oracle can confirm.

**Look for:** in egocentric video people overwhelmingly look at their own hands and the
object they are manipulating. Yellow = NEW, red = OLD.

In [ ]:
K = min(8, len(paths))
idx = np.linspace(0, len(paths)-1, K).astype(int)

fig, ax = plt.subplots(2, 4, figsize=(17, 9))
for a, i in zip(ax.ravel(), idx):
    img = cv2.cvtColor(cv2.imread(paths[i]), cv2.COLOR_BGR2RGB)
    hh, ww = img.shape[:2]
    j = int(np.clip(np.searchsorted(g_ts, f_ts[i]), 0, len(g_ts)-1))
    xn, yn, o1 = project_new(g_yaw[j], g_pit[j])
    xo, yo, o2 = project_old(g_yaw[j], g_pit[j])
    a.imshow(img)
    a.scatter([xo*ww], [yo*hh], s=180, ec="red",    fc="none", lw=2, label="OLD")
    a.scatter([xn*ww], [yn*hh], s=180, ec="yellow", fc="none", lw=2.5, label="NEW")
    a.set_title(f"t={f_ts[i]/1e6:.0f}s" + ("  [out-of-FOV]" if o1 else ""), fontsize=9)
    a.axis("off")
ax.ravel()[0].legend(loc="upper right", fontsize=8)
plt.suptitle("gaze dot: NEW (yellow) vs OLD (red)")
plt.tight_layout(); plt.show()

In [ ]:
# T5b -- WHICH rotation is right? Renders all four candidates side by side.
# Pick the column where the dot consistently lands on the hands / the manipulated object.
CANDIDATES = {
    "none  (x, y)":      lambda x, y: (x, y),
    "cw90  (1-y, x)":    lambda x, y: (1.0 - y, x),      # <- expected: np.rot90(img, k=3)
    "ccw90 (y, 1-x)":    lambda x, y: (y, 1.0 - x),
    "180   (1-x, 1-y)":  lambda x, y: (1.0 - x, 1.0 - y),
}

rows_idx = np.linspace(0, len(paths)-1, 3).astype(int)
fig, ax = plt.subplots(len(rows_idx), len(CANDIDATES), figsize=(4.2*len(CANDIDATES), 4.4*len(rows_idx)))
ax = np.atleast_2d(ax)

for r, i in enumerate(rows_idx):
    img = cv2.cvtColor(cv2.imread(paths[i]), cv2.COLOR_BGR2RGB)
    hh, ww = img.shape[:2]
    j = int(np.clip(np.searchsorted(g_ts, f_ts[i]), 0, len(g_ts)-1))
    xn, yn, _ = project_new(g_yaw[j], g_pit[j], rotate=False)   # native coords
    for c_, (name, fn) in enumerate(CANDIDATES.items()):
        a = ax[r, c_]
        xx, yy_ = fn(xn, yn)
        a.imshow(img)
        a.scatter([xx*ww], [yy_*hh], s=200, ec="yellow", fc="none", lw=3)
        a.axis("off")
        if r == 0:
            a.set_title(name, fontsize=11)
        if c_ == 0:
            a.text(-0.05, 0.5, f"t={f_ts[i]/1e6:.0f}s", transform=a.transAxes,
                   rotation=90, va="center", ha="right", fontsize=9)
plt.suptitle("which rotation puts the dot on what the wearer is looking at?", fontsize=13)
plt.tight_layout(); plt.show()

print("Set ROTATE_CW90 / the config `rotate_cw90` to match the winning column.")
print("Expected winner: cw90 -- it is the coordinate form of np.rot90(img, k=3),")
print("the rotation projectaria_tools prescribes for making Aria RGB frames upright.")

## T6 — Scanpath on a single frame

All gaze samples in a 10 s window drawn over the middle frame. A believable scanpath sits
on objects and clusters into fixations; a broken one drifts off-frame or hugs a border.

In [ ]:
mid = len(paths)//2
img = cv2.cvtColor(cv2.imread(paths[mid]), cv2.COLOR_BGR2RGB)
hh, ww = img.shape[:2]

t0_us, t1_us = f_ts[mid], f_ts[mid] + 10e6
sel = (g_ts >= t0_us) & (g_ts < t1_us) & good
pts = np.array([project_new(y, p)[:2] for y, p in zip(g_yaw[sel], g_pit[sel])])

plt.figure(figsize=(7,7))
plt.imshow(img)
plt.plot(pts[:,0]*ww, pts[:,1]*hh, "-o", ms=4, lw=1, color="yellow", mec="black", mew=.4)
plt.scatter([pts[0,0]*ww], [pts[0,1]*hh], s=150, c="lime", ec="black", zorder=5, label="start")
plt.axis("off"); plt.legend()
plt.title(f"scanpath, {pts.shape[0]} samples over 10 s — t={t0_us/1e6:.0f}s")
plt.show()

## 11 — Stage 2b: DINOv2 encode + gaze patch cell

`frame_features` and `patch_token_at_gaze` verbatim from `src/common/encoders.py`.
The cyan box shows exactly which of the 49 cells the gaze token is taken from —
it must contain the yellow dot.

In [ ]:
import torch, torch.nn.functional as F

DEV = "cuda" if torch.cuda.is_available() else "cpu"
DINO_SIZE = 98        # matches configs/temporal_analysis.yaml -> dino_input_patch
print("device:", DEV)

dino = torch.hub.load("facebookresearch/dinov2", "dinov2_vits14").to(DEV).eval()
for p_ in dino.parameters():
    p_.requires_grad_(False)


@torch.no_grad()
def frame_features(frame_rgb, device, dino_size=DINO_SIZE):
    t = torch.from_numpy(frame_rgb.copy()).permute(2,0,1).float().unsqueeze(0).to(device)/255.0
    t = F.interpolate(t, size=(dino_size, dino_size), mode="bilinear", align_corners=False)
    out = dino.forward_features(t)
    cls = F.normalize(out["x_norm_clstoken"][0], dim=-1)
    patches = F.normalize(out["x_norm_patchtokens"][0], dim=-1)
    grid = int(round(patches.shape[0] ** 0.5))
    return cls, patches, grid

def patch_token_at_gaze(patches, grid, gaze_xy_norm):
    gx = min(grid-1, int(np.clip(gaze_xy_norm[0], 0, 1) * grid))
    gy = min(grid-1, int(np.clip(gaze_xy_norm[1], 0, 1) * grid))
    return patches[gy*grid + gx]

def cosine(a, b):
    return float(F.cosine_similarity(a.unsqueeze(0), b.unsqueeze(0)).item())


cls_l, pat_l, gxy = [], [], []
t0 = time.time()
for p_, t_ in zip(paths, f_ts):
    frame = cv2.cvtColor(cv2.imread(p_), cv2.COLOR_BGR2RGB)
    j = int(np.clip(np.searchsorted(g_ts, t_), 0, len(g_ts)-1))
    x_, y_, o_ = project_new(g_yaw[j], g_pit[j])
    c, pt, GRID = frame_features(frame, DEV)
    cls_l.append(c); pat_l.append(pt); gxy.append((x_, y_, o_))
print(f"encoded {len(paths)} frames in {time.time()-t0:.1f}s   grid={GRID}x{GRID} ({GRID*GRID} tokens)")

sims = []
for i in range(1, len(paths)):
    fs = cosine(cls_l[i-1], cls_l[i])
    p0 = patch_token_at_gaze(pat_l[i-1], GRID, gxy[i-1][:2])
    p1 = patch_token_at_gaze(pat_l[i],   GRID, gxy[i][:2])
    sims.append((fs, cosine(p0, p1)))
sims = np.array(sims)
print(f"\nframe_similarity      : mean {sims[:,0].mean():.3f}  min {sims[:,0].min():.3f}  max {sims[:,0].max():.3f}")
print(f"gaze_patch_token_sim  : mean {sims[:,1].mean():.3f}  min {sims[:,1].min():.3f}  max {sims[:,1].max():.3f}")

In [ ]:
# the cyan cell must contain the yellow dot -- this is the gaze->token binding
k = min(3, len(paths)-1)
fig, ax = plt.subplots(k, 2, figsize=(10, 5*k))
ax = np.atleast_2d(ax)
for r in range(k):
    for c_, i in enumerate([r, r+1]):
        a = ax[r, c_]
        img = cv2.cvtColor(cv2.imread(paths[i]), cv2.COLOR_BGR2RGB)
        hh, ww = img.shape[:2]
        gxn, gyn, _ = gxy[i]
        a.imshow(img)
        a.scatter([gxn*ww], [gyn*hh], s=170, ec="yellow", fc="none", lw=2.5)
        cx = min(GRID-1, int(np.clip(gxn,0,1)*GRID)); cy = min(GRID-1, int(np.clip(gyn,0,1)*GRID))
        a.add_patch(mpatches.Rectangle((cx*ww/GRID, cy*hh/GRID), ww/GRID, hh/GRID,
                                       fill=False, ec="cyan", lw=2.5))
        a.set_title(f"t={f_ts[i]/1e6:.0f}s   cell=({cx},{cy})", fontsize=9); a.axis("off")
    ax[r,0].set_ylabel(f"frame_sim={sims[r,0]:.3f}\npatch_sim={sims[r,1]:.3f}")
plt.suptitle("gaze dot (yellow) inside its DINOv2 cell (cyan)")
plt.tight_layout(); plt.show()

## Stage 4 — Visual verification, sampled by similarity

Everything above sampled frames at **fixed intervals**, which shows you whatever happened
to be there. This section instead picks pairs **by their scores**, so you can look at each
case on purpose:

| Cell | What you get |
|---|---|
| **S4a** | scatter of all pairs, split into the four gate quadrants, plus ranked tables |
| **S4b** | the four **extremes**: most / least similar whole frame, most / least similar gaze token |
| **S4c** | one representative pair for each of the four **gate quadrants** |

Rendering uses `show_pair` **verbatim from `src/common/encoders.py`** — so this is a real
test of Stage 4, not a reimplementation. Two deliberate deviations, both noted in the code:

- the `matplotlib.use("Agg")` line is dropped so figures appear inline
- the gaze crop defaults to the **true DINOv2 cell size** (`w / GRID` ≈ 201 px) instead of
  the repo's `patch_px=64`. The repo's 64 px crop shows a region **10× smaller in area**
  than the token actually covers, which makes the patch look far more precise than it is

**What to look for in each figure:** the yellow circle is the projected gaze; the cyan box
is the DINOv2 cell whose token is compared; the bottom row is that region zoomed. A high
`gaze_patch_token_sim` should visibly show the same thing in both crops, and a low one
should visibly show different things. If they disagree, either the projection or the
similarity is wrong.

The most interesting quadrant is **HIGH frame + LOW gaze** — the scene looks unchanged but
the attended region changed. That case is the entire justification for having two
thresholds instead of one, so it is worth checking that real examples of it exist and look
sensible.

In [ ]:
# ---- verbatim from src/common/encoders.py, except:
#      - the matplotlib.use("Agg") line is dropped so figures render inline in Colab
#      - a `suptitle` argument is added to label which case is being shown
def gaze_crop_rgb(frame, g, patch_px=64):
    g = np.where(np.isfinite(g), g, 0.5)
    h, w = frame.shape[:2]; cx, cy = g[0]*w, g[1]*h; half = patch_px//2
    x0,x1 = int(np.clip(cx-half,0,w-1)), int(np.clip(cx+half,0,w))
    y0,y1 = int(np.clip(cy-half,0,h-1)), int(np.clip(cy+half,0,h))
    c = frame[y0:y1, x0:x1]
    if c.shape[:2] != (patch_px, patch_px): c = cv2.resize(c, (patch_px, patch_px))
    return c


def show_pair(frame_a, frame_b, gaze_a, gaze_b, frame_sim_, patch_sim_,
              patch_px=64, save_path=None, grid=None, suptitle=None):
    crop_a = gaze_crop_rgb(frame_a, gaze_a, patch_px)
    crop_b = gaze_crop_rgb(frame_b, gaze_b, patch_px)
    fig, ax = plt.subplots(2, 2, figsize=(9, 9))
    for a, img, g, t in [(ax[0,0], frame_a, gaze_a, "frame t-1"),
                         (ax[0,1], frame_b, gaze_b, "frame t")]:
        h, w = img.shape[:2]
        a.imshow(img)
        a.scatter([g[0]*w], [g[1]*h], s=160, ec='yellow', fc='none', lw=2)
        if grid:   # the DINOv2 cell whose token is actually compared
            gx = min(grid-1, int(np.clip(g[0],0,1)*grid)); gy = min(grid-1, int(np.clip(g[1],0,1)*grid))
            a.add_patch(mpatches.Rectangle((gx*w/grid, gy*h/grid), w/grid, h/grid,
                                           fill=False, ec='cyan', lw=2))
        a.set_title(t); a.axis('off')
    ax[1,0].imshow(crop_a); ax[1,0].set_title("gaze patch t-1"); ax[1,0].axis('off')
    ax[1,1].imshow(crop_b); ax[1,1].set_title("gaze patch t"); ax[1,1].axis('off')
    fig.suptitle(suptitle or f"frame_sim={frame_sim_:.3f}   gaze_patch_token_sim={patch_sim_:.3f}")
    plt.tight_layout()
    if save_path: plt.savefig(save_path, dpi=120)
    plt.show()


def render_pair(k, label, patch_px=None):
    """Render consecutive pair (k, k+1) with the repo's own show_pair."""
    fa = cv2.cvtColor(cv2.imread(paths[k]),   cv2.COLOR_BGR2RGB)
    fb = cv2.cvtColor(cv2.imread(paths[k+1]), cv2.COLOR_BGR2RGB)
    ga = np.array(gxy[k][:2]);  gb = np.array(gxy[k+1][:2])
    # NOTE: the repo default is patch_px=64, but one DINOv2 cell is w/GRID px
    # (~201 px at GRID=7). 64 px shows a far SMALLER region than the token covers,
    # so the crop here defaults to the true cell size to match what is compared.
    pp = patch_px or int(fa.shape[1] / GRID)
    show_pair(fa, fb, ga, gb, sims[k,0], sims[k,1], patch_px=pp, grid=GRID,
              suptitle=(f"{label}\nt={f_ts[k]/1e6:.0f}s -> {f_ts[k+1]/1e6:.0f}s        "
                        f"frame_sim={sims[k,0]:.3f}   gaze_patch_token_sim={sims[k,1]:.3f}"))

print(f"ready -- {len(sims)} pairs, GRID={GRID}x{GRID}, one cell = {int(1408/GRID)} px")

In [ ]:
# S4a -- where do the pairs actually sit? Pick split thresholds and see the quadrants.
frame_sim = sims[:, 0]
patch_sim = sims[:, 1]

# Medians are used purely so every quadrant is populated for inspection.
# The real gate would use tuned thresholds -- override these to try your own.
TH_FRAME = float(np.median(frame_sim))
TH_GAZE  = float(np.median(patch_sim))

print(f"{len(sims)} consecutive pairs\n")
print(f"frame_similarity     : mean {frame_sim.mean():.3f}  std {frame_sim.std():.3f}  "
      f"range [{frame_sim.min():.3f}, {frame_sim.max():.3f}]")
print(f"gaze_patch_token_sim : mean {patch_sim.mean():.3f}  std {patch_sim.std():.3f}  "
      f"range [{patch_sim.min():.3f}, {patch_sim.max():.3f}]")

r = np.corrcoef(frame_sim, patch_sim)[0, 1]
print(f"\ncorrelation between the two signals: {r:+.3f}")
print("   near 0  -> the gaze signal carries information the frame signal does not (good:")
print("             two thresholds are justified)")
print("   near 1  -> the gaze signal is redundant, one threshold would do")

print(f"\nsplit thresholds (medians): frame {TH_FRAME:.3f}   gaze {TH_GAZE:.3f}\n")
hi_f, hi_g = frame_sim >= TH_FRAME, patch_sim >= TH_GAZE
for nm, mk, act in [("HIGH frame + HIGH gaze", hi_f & hi_g,  "DISCARD"),
                    ("HIGH frame + LOW  gaze", hi_f & ~hi_g, "SEND (subtle attended change)"),
                    ("LOW  frame + HIGH gaze", ~hi_f & hi_g, "SEND"),
                    ("LOW  frame + LOW  gaze", ~hi_f & ~hi_g,"SEND")]:
    print(f"   {nm}: {mk.sum():3d} pairs   -> {act}")

# ranked tables so you can pick other pairs to inspect by hand
o = np.argsort(frame_sim)
print("\n5 LEAST similar frames (pair idx, frame_sim, gaze_sim):")
for k in o[:5]:  print(f"   {k:3d}   {frame_sim[k]:.3f}   {patch_sim[k]:.3f}   t={f_ts[k]/1e6:.0f}s")
print("5 MOST similar frames:")
for k in o[-5:]: print(f"   {k:3d}   {frame_sim[k]:.3f}   {patch_sim[k]:.3f}   t={f_ts[k]/1e6:.0f}s")

fig, ax = plt.subplots(1, 2, figsize=(13, 5))
ax[0].scatter(frame_sim, patch_sim, s=30, alpha=.75, edgecolor="k", linewidth=.3)
ax[0].axvline(TH_FRAME, color="k", lw=.9, ls="--"); ax[0].axhline(TH_GAZE, color="k", lw=.9, ls="--")
ax[0].set_xlabel("frame_similarity (whole scene)"); ax[0].set_ylabel("gaze_patch_token_sim (attended region)")
ax[0].set_title("every consecutive pair, split into gate quadrants")
for xf, yf, txt, col in [(.02,.96,"LOW frame / HIGH gaze\nSEND","tab:orange"),
                         (.62,.96,"HIGH / HIGH\nDISCARD","tab:green"),
                         (.02,.04,"LOW / LOW\nSEND","tab:red"),
                         (.62,.04,"HIGH frame / LOW gaze\nSEND","tab:purple")]:
    ax[0].text(xf, yf, txt, transform=ax[0].transAxes, fontsize=8, color=col,
               va="top" if yf > .5 else "bottom", weight="bold")
ax[1].hist(frame_sim, bins=25, alpha=.65, label="frame_similarity")
ax[1].hist(patch_sim, bins=25, alpha=.65, label="gaze_patch_token_sim")
ax[1].set_xlabel("cosine similarity"); ax[1].set_ylabel("count"); ax[1].legend()
ax[1].set_title("distributions -- gaze should be lower and wider")
plt.tight_layout(); plt.show()

In [ ]:
# S4b -- the four EXTREMES: most/least similar overall frame, most/least similar gaze token.
# These are the pairs that define the ends of each scale.
EXTREMES = [
    ("MOST similar FRAMES   (max frame_similarity)",      int(np.argmax(frame_sim))),
    ("LEAST similar FRAMES  (min frame_similarity)",      int(np.argmin(frame_sim))),
    ("MOST similar GAZE     (max gaze_patch_token_sim)",  int(np.argmax(patch_sim))),
    ("LEAST similar GAZE    (min gaze_patch_token_sim)",  int(np.argmin(patch_sim))),
]

for label, k in EXTREMES:
    print(f"\n{'='*78}\n### {label}   -> pair {k}  (t={f_ts[k]/1e6:.0f}s -> {f_ts[k+1]/1e6:.0f}s)")
    print(f"    frame_sim={frame_sim[k]:.3f}   gaze_patch_token_sim={patch_sim[k]:.3f}\n")
    render_pair(k, label)

In [ ]:
# S4c -- the four GATE quadrants from the FilterFrameForVLM rule.
# One representative pair per quadrant, chosen as the most extreme corner case.
hi_f, hi_g = frame_sim >= TH_FRAME, patch_sim >= TH_GAZE

QUADRANTS = [
    ("HIGH frame + HIGH gaze  ->  DISCARD",
     "scene stable AND attended region stable: reuse cached features",
     frame_sim + patch_sim,           "max",  hi_f &  hi_g),
    ("HIGH frame + LOW gaze   ->  SEND",
     "scene looks the same but what they are LOOKING AT changed -- the subtle case "
     "a single global threshold would miss",
     frame_sim - patch_sim,           "max",  hi_f & ~hi_g),
    ("LOW frame + HIGH gaze   ->  SEND",
     "scene changed a lot but the attended object stayed the same (tracking while moving)",
     patch_sim - frame_sim,           "max", ~hi_f &  hi_g),
    ("LOW frame + LOW gaze    ->  SEND",
     "everything changed: head turn, room change, or a large saccade",
     frame_sim + patch_sim,           "min", ~hi_f & ~hi_g),
]

for title, why, score, how, mask in QUADRANTS:
    if not mask.any():
        print(f"\n### {title}\n   (no pairs in this quadrant for this sequence)\n")
        continue
    s = np.where(mask, score, -np.inf if how == "max" else np.inf)
    k = int(np.argmax(s) if how == "max" else np.argmin(s))
    print(f"\n{'='*78}\n### {title}      ({mask.sum()} pairs in this quadrant)\n   {why}\n")
    render_pair(k, title)

## 12 — Verdict

In [ ]:
print(f"sequence: {SEQ}\n")
checks = [
    ("Stage 0  downloads + unzip",        GAZE_CSV is not None and os.path.exists(mp4) and os.path.exists(vrs_path)),
    ("clock    video vs gaze duration",   clock_gap < 1.0),
    ("Stage 1  frames subsampled",        len(paths) > 1),
    ("T1       matches Meta oracle",      T1_PASS),
    ("T3       yaw->x, pitch->y",         T3_PASS),
    ("T4       distribution sane",        T4_PASS),
    ("Stage 2  DINOv2 encoded",           len(cls_l) == len(paths)),
]
for name, ok_ in checks:
    print(f"   [{'PASS' if ok_ else 'FAIL'}]  {name}")

print(f"\n   old-code error was {d_old.mean()*W_cal:.1f} px "
      f"({d_old.mean()*W_cal/(W_cal/7):.2f} DINOv2 cells) -- now {d_new.mean()*W_cal:.4f} px")
print("\nT5/T6 are visual: confirm the yellow dot sits on hands/objects, not beside them.")

---

## Reading the results

| Outcome | Meaning |
|---|---|
| T1 PASS, T3 PASS, T5/T5b look right | projection is correct — proceed to build the CSV |
| T1 PASS but T3 FAIL | transform chain is right, rotation is wrong → pick the winning column in T5b and set `ROTATE_CW90` / config `rotate_cw90` to match |
| T1 FAIL | the transform chain still disagrees with Meta's helper — check that `get_transform_device_cpf()` and `get_transform_device_camera()` exist in your `projectaria_tools` version |
| T3 PASS but T5 looks off | remaining suspects: the frame↔gaze clock offset, or `DEPTH_M` being wrong for near objects |
| T4 high out-of-FOV | wrong stream label, or the wearer genuinely looked outside the RGB FOV a lot |

**Do not trust T4 alone.** On the first run it PASSED while the coordinates were rotated
90° — distribution statistics cannot detect a rotation. T3 and T5b are the tests that
earn their keep here.

The repo now matches this notebook: `src/common/gaze_geometry.py` uses the tutorial chain
plus `native_to_upright_cw90`, controlled by `rotate_cw90` and `gaze_depth_m` in
`configs/temporal_analysis.yaml`.

Once green, run the real thing:

```bash
python -m src.temporal.build_similarity_csv \
  --urls_json  <your.json> \
  --seqs       <SEQ> \
  --raw_dir    /content/aea_check/raw \
  --frames_dir /content/aea_check/frames_1fps \
  --out_csv    /content/aea_check/frame_vs_gaze_similarity.csv
```